In [ ]:
from pyspark.sql import SparkSession
import pyspark.sql.types as st
import pyspark.sql.functions as sf

In [ ]:
spark = (
    SparkSession
    .builder
    .appName("Pyspark tutorial")
    .master("local[*]")
    .getOrCreate()
)

#### Read / Write

In [ ]:
data = [
    (1, "Alice", 20),
    (2, "Bob", 25),
    (3, "Charlie", 30),
    (4, "David", 35)
]

columns = ["id","name","age"]

df = spark.createDataFrame(data,columns)

df.show()

In [ ]:
df.printSchema()

In [ ]:
schema = st. StructType (
    [
        st. StructField("id", st.IntegerType(),True),
        st. StructField("name", st.StringType(),True),
        st. StructField("age", st.IntegerType(),True),
    ]

)


df = spark.createDataFrame(data,schema=schema)

df.printSchema()

In [ ]:
df_read_csv = spark.read.csv("C:\\Users\\win11\\Documents\\DataEngineer_kepzes\\Codes\\src\\pyspark_project\\src\\data\\2010-summary.csv", header=True)

df_read_csv.show()

In [ ]:
df_read_csv = (
    spark
    .read
    .format("csv")
    .option("header",True)
    .option("delimiter",",")
    .load("C:\\Users\\win11\\Documents\\DataEngineer_kepzes\\Codes\\src\\pyspark_project\\src\\data\\2010-summary.csv")

)

df_read_csv.show(2)

In [ ]:
df_read_parquet = (
    spark
    .read
    .format("parquet")
    .load("C:\\Users\\win11\\Documents\\DataEngineer_kepzes\\Codes\\src\\pyspark_project\\src\\data\\yellow_tripdata_2024-09.parquet")
)

df_read_parquet.show(10)

In [ ]:
df_read_parquet.printSchema()

In [ ]:
df_read_parquet.count()

#### Transformations
##### Selecting

In [ ]:
taxi_df = (
    spark
    .read
    .format("parquet")
    .load("C:\\Users\\win11\\Documents\\DataEngineer_kepzes\\Codes\\src\\pyspark_project\\src\\data\\yellow_tripdata_2024-09.parquet")
)

In [ ]:
taxi_df.describe().show()

In [ ]:
taxi_df.select(sf.col("VendorID"),sf.col("passenger_count")).show(5)

##### Filtering

In [20]:
taxi_df.where(sf.col("trip_distance") > 5).show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       1| 2024-09-01 00:05:51|  2024-09-01 00:45:03|              1|          9.8|         1|                 N|         138|          48|           1|       47.8|10.25|    0.5|      13.

In [28]:
(
taxi_df
.where (
    
    (sf.col("trip_distance") > 5)
     & (sf.col("passenger_count") > 1)
     
).show(5)

)



+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       1| 2024-09-01 00:08:28|  2024-09-01 00:39:06|              4|          9.8|         1|                 N|          93|         161|           1|       44.3|  3.5|    0.5|      9.8

##### WithColumn

In [ ]:
taxi_df = (
    taxi_df
    .withColumn(
        "total_amount_with_all_tax",
        sf.col("total_amount") + sf.col("congestion_surcharge") + sf.col("Airport_fee")
    )
    .withColumn("total_tip_percentage",sf.format_number(sf.col("total_tip_percentage"),2))

)

NameError: name 'taxi_df' is not defined

In [30]:
taxi_df.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+-------------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|total_amount_with_all_tax|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+-------------------------+
|       1| 2024-09-01 00:05:51|  2024-09-01 00:45:03|              1|          9.8|         1|               

##### WithColumnRename